In [1]:
import pandas as pd

reviews_file = "reviews_with_predictions.csv"
metadata_file = "Toys_and_Games/meta_Toys_and_Games.jsonl"
output_file = "reviews_with_products.csv"

# Load the review predictions
reviews = pd.read_csv(reviews_file)

# Load metadata stored as JSON Lines (one JSON object per line)
metadata = pd.read_json(metadata_file, lines=True)


In [2]:
print("Review rows:", len(reviews))
print("Metadata rows:", len(metadata))
print("Review columns:", reviews.columns.tolist())
print("Metadata columns:", metadata.columns.tolist())


Review rows: 147640
Metadata rows: 890874
Review columns: ['rating', 'title', 'text', 'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase', 'user_id', 'review_text', 'sentiment', 'predicted_sentiment', 'prediction_confidence', 'rating_sentiment', 'sentiment_disagreement']
Metadata columns: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']


In [3]:
# Check the join key exists in both datasets
assert "parent_asin" in reviews.columns
assert "parent_asin" in metadata.columns


In [4]:
# Standardise IDs to avoid mismatches caused by whitespace or types
reviews["parent_asin"] = (
    reviews["parent_asin"].astype("string").str.strip()
)
metadata["parent_asin"] = (
    metadata["parent_asin"].astype("string").str.strip()
)

In [5]:
# Rename overlapping metadata fields to distinguish them from review fields
metadata = metadata.rename(columns={
    "title": "product_title",
    "average_rating": "product_average_rating",
    "rating_number": "product_rating_count",
})

In [6]:
# Inspect duplicate product IDs before joining
duplicate_count = metadata["parent_asin"].duplicated().sum()
print("Duplicate metadata IDs:", duplicate_count)

Duplicate metadata IDs: 0


In [7]:
# Keep one metadata row per product ID for a many-reviews-to-one-product join.
# Inspect duplicates first if they have conflicting product details.
metadata = metadata.drop_duplicates(
    subset="parent_asin",
    keep="first"
)

In [8]:
# Match reviews to products by ID, never by row position
joined = reviews.merge(
    metadata,
    on="parent_asin",
    how="left",
    validate="many_to_one",
    indicator=True,
)

In [9]:
print("\nJoin results:")
print(joined["_merge"].value_counts())

match_rate = joined["product_title"].notna().mean() * 100
print(f"Product-title match rate: {match_rate:.2f}%")


Join results:
_merge
both          147640
left_only          0
right_only         0
Name: count, dtype: int64
Product-title match rate: 100.00%


In [10]:
# Save all review and metadata columns
joined.drop(columns="_merge").to_csv(
    output_file,
    index=False
)

print(f"Saved {len(joined):,} reviews to {output_file}")

Saved 147,640 reviews to reviews_with_products.csv


In [11]:
joined[[
    "parent_asin",
    "product_title",
    "price",
    "product_average_rating",
    "predicted_sentiment",
    "review_text",
]].head(10)

,parent_asin,product_title,price,product_average_rating,predicted_sentiment,review_text
0,B09QH7QJS7,"Schleich bayala, 5-Piece Starter Set with Fair...",34.44,4.6,positive,Granddaughters love them!. I purchased several...
1,B06XYKSKQP,Liberty Imports Electronic Walking Jurassic St...,14.97,4.3,neutral,"Arrived broken. It’s cute, but it arrived brok..."
2,B07XRSD5R9,Pokémon XY Evolutions Bundle,NaN,4.7,positive,Dylan loves them!!!. Bough for my grandson Dyl...
3,B007JWWUDW,Janod Story Train Circus,NaN,4.5,positive,Janod quality and very cute.... This was great...
4,B00MZG6OO8,"maxim enterprise, inc. Wooden Animal Train Cir...",14.99,4.3,neutral,I used for my daughters circus birthday party ...
5,B00SO7HF6I,Fisher-Price Laugh & Learn Sis' Smart Stages P...,44.99,4.5,neutral,Cute idea but just OK. I love the laugh and le...
6,B00TLEMSVK,Hasbro Gaming Candy Land Board Game,NaN,4.5,positive,"❤. Candyland, old favorite- what's not to love!"
7,B006GBITXC,Fisher-Price Octonauts Peso Plush,NaN,4.2,neutral,Cute but small.... Cute but much smaller than ...
8,B01N0UJMXF,Play-Doh Kitchen Creations Cookout Creations P...,NaN,4.7,positive,Hours of fun. This is great! I chose this over...
9,B0B4TML5B1,"Aurora World Ballerina Mia Doll Plush, 14.5 in...",NaN,4.6,positive,Love!. My 3 year old daughter loves this soft ...


Sanity Checks

In [14]:
joined.head(5)

,rating,title,text,parent_asin,timestamp,helpful_vote,verified_purchase,user_id,review_text,sentiment,...,price,images,videos,store,categories,details,bought_together,subtitle,author,_merge
0,5.0,Granddaughters love them!,I purchased several of these for my granddaugh...,B09QH7QJS7,1970-01-01 00:27:57.939664713,0,True,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,Granddaughters love them!. I purchased several...,positive,...,34.44,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Schleich: Where Stories Begin', 'u...",Schleich,"[Toys & Games, Toy Figures & Playsets, Playsets]",{'Product Dimensions': '9.6 x 9.6 x 7.4 inches...,NaN,NaN,NaN,both
1,3.0,Arrived broken,"It’s cute, but it arrived broken & has some pr...",B06XYKSKQP,1970-01-01 00:27:19.855230760,1,True,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,"Arrived broken. It’s cute, but it arrived brok...",neutral,...,14.97,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Broken dinosaur', 'url': 'https://...",Liberty Imports,"[Toys & Games, Toy Figures & Playsets, Action ...",{'Product Dimensions': '9.7 x 6.7 x 4.7 inches...,NaN,NaN,NaN,both
2,5.0,Dylan loves them!!!,Bough for my grandson Dylan. He loves them.,B07XRSD5R9,1970-01-01 00:26:20.949719154,0,True,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,Dylan loves them!!!. Bough for my grandson Dyl...,positive,...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Pokemon,"[Toys & Games, Games & Accessories, Card Games...","{'Product Dimensions': '8 x 4 x 5 inches', 'It...",NaN,NaN,NaN,both
3,5.0,Janod quality and very cute...,This was great for my daughters circus themed ...,B007JWWUDW,1970-01-01 00:24:31.542588000,0,True,AGKASBHYZPGTEPO6LWZPVJWB2BVA,Janod quality and very cute.... This was great...,positive,...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Janod Memonimo Memory Game', 'url'...",Janod,"[Toys & Games, Preschool, Pre-Kindergarten Toy...",{'Product Dimensions': '12.6 x 1.18 x 3.74 inc...,NaN,NaN,NaN,both
4,3.0,I used for my daughters circus birthday party ...,This is very cute but beware that the animals ...,B00MZG6OO8,1970-01-01 00:24:31.541996000,2,True,AGKASBHYZPGTEPO6LWZPVJWB2BVA,I used for my daughters circus birthday party ...,neutral,...,14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],"maxim enterprise, inc.","[Toys & Games, Preschool, Pre-Kindergarten Toy...",{'Product Dimensions': '8.9 x 4.3 x 1.8 inches...,NaN,NaN,NaN,both


In [16]:
# Check how many reviews matched product metadata
print("Total reviews:", len(joined))

print(
    "Matched products:",
    joined["product_title"].notna().sum()
)

print(
    "Unmatched products:",
    joined["product_title"].isna().sum()
)

print(
    "Match rate:",
    round(joined["product_title"].notna().mean() * 100, 2),
    "%"
)

print(joined["_merge"].value_counts())

Total reviews: 147640
Matched products: 147640
Unmatched products: 0
Match rate: 100.0 %
_merge
both          147640
left_only          0
right_only         0
Name: count, dtype: int64


In [17]:
metadata_columns = [
    "product_title",
    "price",
    "details",
    "categories",
    "store",
    "bought_together",
    "subtitle",
    "author",
]

print(
    joined[metadata_columns]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .rename("Missing (%)")
)

subtitle           100.00
author             100.00
bought_together    100.00
price               33.27
store                0.14
categories           0.00
product_title        0.00
details              0.00
Name: Missing (%), dtype: float64
